In [9]:
# SECTION 1: boto3 BASICS

In [10]:
from dotenv import load_dotenv
import os
load_dotenv()
AWS_REGION       = os.getenv("AWS_REGION", "us-east-1")
SQS_QUEUE_URL    = os.getenv("SQS_QUEUE_URL")          # Full queue URL
AWS_ACCESS_KEY   = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY   = os.getenv("AWS_SECRET_ACCESS_KEY")

In [11]:
import boto3
from botocore.exceptions import ClientError, NoCredentialsError

In [12]:
# HIGH-LEVEL RESOURCE: object-oriented interface, easier for most tasks
s3_resource = boto3.resource("s3", region_name="us-east-2")

In [14]:
# LOW-LEVEL CLIENT: gives access to all raw API calls, returns dicts
s3_client = boto3.client(
    "s3",
    region_name="us-east-2",
    # Explicitly passing creds (use env vars or IAM roles in production)
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
)

In [15]:
response = s3_client.list_buckets()

In [17]:
response

{'ResponseMetadata': {'RequestId': 'VCHBDFB1S0ATW0WS',
  'HostId': 'RRqzn+8oaXHGzOnnJbfAM7tJMW4HhYSFGacSdO2G8k2PHEqKqdZqYCi6bqAwGawLslRtTGm1I4ljlm+eemZZzk0bJ6vEh78Y',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'RRqzn+8oaXHGzOnnJbfAM7tJMW4HhYSFGacSdO2G8k2PHEqKqdZqYCi6bqAwGawLslRtTGm1I4ljlm+eemZZzk0bJ6vEh78Y',
   'x-amz-request-id': 'VCHBDFB1S0ATW0WS',
   'date': 'Thu, 05 Mar 2026 03:34:08 GMT',
   'content-type': 'application/xml',
   'transfer-encoding': 'chunked',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'Buckets': [{'Name': 'de-aws-snowflake-demo',
   'CreationDate': datetime.datetime(2026, 2, 26, 3, 44, 3, tzinfo=tzutc()),
   'BucketArn': 'arn:aws:s3:::de-aws-snowflake-demo'},
  {'Name': 'de-mwaa-202603',
   'CreationDate': datetime.datetime(2026, 3, 1, 11, 46, tzinfo=tzutc()),
   'BucketArn': 'arn:aws:s3:::de-mwaa-202603'},
  {'Name': 'de-mwaa-202603-ohio',
   'CreationDate': datetime.datetime(2026, 3, 1, 11, 49, 11, tzinfo=tzutc()),
   'BucketArn': 'arn:aws:

In [18]:
# --- Listing S3 buckets ---
def list_buckets():
    """List all S3 buckets in your account."""
    response = s3_client.list_buckets()
    buckets = [b["Name"] for b in response.get("Buckets", [])]
    print("Buckets:", buckets)
    return buckets

In [19]:
list_buckets()

Buckets: ['de-aws-snowflake-demo', 'de-mwaa-202603', 'de-mwaa-202603-ohio']


['de-aws-snowflake-demo', 'de-mwaa-202603', 'de-mwaa-202603-ohio']

In [20]:
# --- Listing objects in a bucket ---
def list_objects(bucket_name, prefix=""):
    """
    List objects in a bucket, optionally filtered by prefix (like a folder path).
    Uses pagination to handle buckets with many objects.
    """
    paginator = s3_client.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

    keys = []
    for page in pages:
        for obj in page.get("Contents", []):
            keys.append(obj["Key"])
            print(f"  Key: {obj['Key']} | Size: {obj['Size']} bytes | Modified: {obj['LastModified']}")
    return keys

In [21]:
list_objects('de-aws-snowflake-demo')

  Key: retail/ | Size: 0 bytes | Modified: 2026-02-27 04:04:37+00:00
  Key: retail/customers.csv | Size: 1035 bytes | Modified: 2026-02-27 04:04:52+00:00
  Key: retail/products.csv | Size: 687 bytes | Modified: 2026-02-27 04:04:52+00:00
  Key: retail/sales.csv | Size: 5019 bytes | Modified: 2026-02-27 04:04:52+00:00
  Key: sales/ | Size: 0 bytes | Modified: 2026-02-26 03:44:50+00:00
  Key: sales/sales_large.csv | Size: 59329 bytes | Modified: 2026-03-03 04:32:04+00:00


['retail/',
 'retail/customers.csv',
 'retail/products.csv',
 'retail/sales.csv',
 'sales/',
 'sales/sales_large.csv']

In [25]:
# --- Checking if a bucket or object exists ---
def object_exists(bucket_name, key):
    """Return True if an object exists in S3, False otherwise."""
    try:
        s3_client.head_object(Bucket=bucket_name, Key=key)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] == "404":
            return False
        raise  # re-raise unexpected errors

In [27]:
object_exists('de-aws-snowflake-demo','retail/')

True

In [8]:
# ─────────────────────────────────────────────
# SECTION 2: READING DATA FROM S3 USING PYTHON
# ─────────────────────────────────────────────

In [35]:
import io
import csv
import json
import pandas as pd

In [30]:
# --- Download a file to local disk ---
def download_file(bucket_name, s3_key, local_path):
    """Download an S3 object to a local file."""
    try:
        s3_client.download_file(bucket_name, s3_key, local_path)
        print(f"Downloaded s3://{bucket_name}/{s3_key} → {local_path}")
    except NoCredentialsError:
        print("ERROR: AWS credentials not found.")
    except ClientError as e:
        print(f"ERROR: {e}")

In [31]:
download_file('de-aws-snowflake-demo','retail/customers.csv','s3_customers.csv')

Downloaded s3://de-aws-snowflake-demo/retail/customers.csv → s3_customers.csv


In [36]:
# --- Read a CSV from S3 directly into memory (no local file) ---
def read_csv_from_s3(bucket_name, s3_key):
    """Read a CSV file from S3 into a pandas DataFrame without saving locally."""
    response = s3_client.get_object(Bucket=bucket_name, Key=s3_key)
    body = response["Body"].read()              # raw bytes
    df = pd.read_csv(io.BytesIO(body))          # wrap bytes in a file-like object
    print(f"Loaded {len(df)} rows from s3://{bucket_name}/{s3_key}")
    return df

In [37]:
cust_df = read_csv_from_s3('de-aws-snowflake-demo','retail/customers.csv')

Loaded 50 rows from s3://de-aws-snowflake-demo/retail/customers.csv


In [39]:
# --- Read a JSON file from S3 ---
def read_json_from_s3(bucket_name, s3_key):
    """Read a JSON file from S3 and return a Python dict/list."""
    response = s3_client.get_object(Bucket=bucket_name, Key=s3_key)
    content = response["Body"].read().decode("utf-8")
    data = json.loads(content)
    return data

In [42]:
# --- Read a Parquet file from S3 ---
def read_parquet_from_s3(bucket_name, s3_key):
    """
    Read a Parquet file from S3.
    Requires: pip install pyarrow
    """
    response = s3_client.get_object(Bucket=bucket_name, Key=s3_key)
    body = response["Body"].read()
    df = pd.read_parquet(io.BytesIO(body))
    return df

In [44]:
# --- Read multiple files matching a prefix (e.g., partitioned data) ---
def read_partitioned_csvs(bucket_name, prefix):
    """
    Read all CSV files under a given S3 prefix and combine them.
    Common pattern for date-partitioned data: data/year=2024/month=01/*.csv
    """
    keys = list_objects(bucket_name, prefix)
    csv_keys = [k for k in keys if k.endswith(".csv")]

    dfs = []
    for key in csv_keys:
        df = read_csv_from_s3(bucket_name, key)
        df["_source_file"] = key  # track which file each row came from
        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True)
    print(f"Combined {len(csv_keys)} files → {len(combined)} total rows")
    return combined


In [45]:
# SECTION 3: WRITING PROCESSED DATA BACK TO S3

In [46]:
# --- Upload a local file to S3 ---
def upload_file(local_path, bucket_name, s3_key):
    """Upload a local file to S3."""
    s3_client.upload_file(local_path, bucket_name, s3_key)
    print(f"Uploaded {local_path} → s3://{bucket_name}/{s3_key}")

In [48]:
# --- Write a DataFrame to S3 as CSV (in-memory, no local file) ---
def write_csv_to_s3(df, bucket_name, s3_key):
    """Serialize a DataFrame to CSV and upload directly to S3."""
    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    s3_client.put_object(
        Bucket=bucket_name,
        Key=s3_key,
        Body=buffer.getvalue().encode("utf-8"),
        ContentType="text/csv",
    )
    print(f"Written {len(df)} rows → s3://{bucket_name}/{s3_key}")

In [49]:
# --- Write a DataFrame to S3 as Parquet (preferred for large data) ---
def write_parquet_to_s3(df, bucket_name, s3_key):
    """
    Write a DataFrame as Parquet to S3.
    Parquet is columnar — much faster for analytics than CSV.
    Requires: pip install pyarrow
    """
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow")
    buffer.seek(0)
    s3_client.put_object(
        Bucket=bucket_name,
        Key=s3_key,
        Body=buffer.read(),
        ContentType="application/octet-stream",
    )
    print(f"Written Parquet ({len(df)} rows) → s3://{bucket_name}/{s3_key}")

In [50]:
# --- Write partitioned output (e.g., by date) ---
def write_partitioned_parquet(df, bucket_name, base_prefix, partition_col="date"):
    """
    Write a DataFrame to S3 partitioned by a column value.
    Output: s3://bucket/base_prefix/date=2024-01-15/data.parquet
    """
    for value, partition_df in df.groupby(partition_col):
        key = f"{base_prefix}/{partition_col}={value}/data.parquet"
        write_parquet_to_s3(partition_df.drop(columns=[partition_col]), bucket_name, key)

In [53]:
# --- Copy an object within or across buckets ---
def copy_object(src_bucket, src_key, dst_bucket, dst_key):
    """Copy an S3 object without downloading it locally."""
    s3_client.copy_object(
        CopySource={"Bucket": src_bucket, "Key": src_key},
        Bucket=dst_bucket,
        Key=dst_key,
    )
    print(f"Copied s3://{src_bucket}/{src_key} → s3://{dst_bucket}/{dst_key}")

In [54]:
copy_object('de-aws-snowflake-demo','retail/customers.csv','de-aws-snowflake-demo','retail_bkp/customers.csv')

Copied s3://de-aws-snowflake-demo/retail/customers.csv → s3://de-aws-snowflake-demo/retail_bkp/customers.csv


In [56]:
# --- Delete an object ---
def delete_object(bucket_name, s3_key):
    s3_client.delete_object(Bucket=bucket_name, Key=s3_key)
    print(f"Deleted s3://{bucket_name}/{s3_key}")

In [57]:
delete_object('de-aws-snowflake-demo','retail_bkp/customers.csv')

Deleted s3://de-aws-snowflake-demo/retail_bkp/customers.csv


In [23]:
# SECTION 4: S3 → PANDAS → SNOWFLAKE FLOW

In [24]:
"""
The standard data engineering flow:
  1. Extract  → Read raw data from S3
  2. Transform → Clean and reshape with pandas
  3. Load     → Write results to Snowflake

Install Snowflake connector:
    pip install snowflake-connector-python[pandas]
"""

'\nThe standard data engineering flow:\n  1. Extract  → Read raw data from S3\n  2. Transform → Clean and reshape with pandas\n  3. Load     → Write results to Snowflake\n\nInstall Snowflake connector:\n    pip install snowflake-connector-python[pandas]\n'

In [60]:
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

In [61]:
def get_snowflake_connection():
    """
    Create and return a Snowflake connection.
    Store credentials in environment variables — never hardcode them.
    """
    conn = snowflake.connector.connect(
        user='detrainings01',
        password='DataEngineering@2026',
        account='sqqpjbk-pi02317',   # e.g. "xy12345.us-east-1"
        warehouse='COMPUTE_WH',
        database="MY_DB1",
        schema='MY_SC1',
        role="SYSADMIN",
    )
    return conn

In [62]:
# --- Run a query and return results as a DataFrame ---
def query_snowflake(conn, sql):
    """Execute a SQL query and return results as a pandas DataFrame."""
    cursor = conn.cursor()
    cursor.execute(sql)
    df = cursor.fetch_pandas_all()
    cursor.close()
    return df

In [63]:
# --- Write a DataFrame to Snowflake ---
def load_to_snowflake(conn, df, table_name, database, schema, overwrite=False):
    """
    Load a pandas DataFrame into a Snowflake table using write_pandas.
    - overwrite=True  → TRUNCATE the table before inserting
    - overwrite=False → APPEND rows to existing table
    Column names in the DataFrame must match the Snowflake table columns.
    """
    # Snowflake expects uppercase column names
    df.columns = [c.upper() for c in df.columns]

    success, nchunks, nrows, _ = write_pandas(
        conn=conn,
        df=df,
        table_name=table_name.upper(),
        database=database.upper(),
        schema=schema.upper(),
        overwrite=overwrite,
        auto_create_table=True,  # creates table if it doesn't exist
    )
    if success:
        print(f"Loaded {nrows} rows into {database}.{schema}.{table_name} ({nchunks} chunk(s))")
    else:
        print("Load failed.")

In [64]:
# --- Pandas transformation examples ---
def transform_sales_data(df):
    """
    Example transformation: clean and enrich a raw sales DataFrame.
    Adapt column names and logic to your actual data.
    """
    # Drop completely empty rows
    df = df.dropna(how="all")

    # Standardize column names: lowercase, spaces → underscores
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    # Parse date column
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

    # Drop rows with unparseable dates
    df = df.dropna(subset=["order_date"])

    # Derived columns
    df["year"]  = df["order_date"].dt.year
    df["month"] = df["order_date"].dt.month
    #df["revenue"] = df["quantity"] * df["unit_price"]
    df["quantity"] =  df["total_amount"] / df["price"]

    # Normalize a text column
    #df["region"] = df["region"].str.strip().str.upper()
    df["category"] = df["category"].str.strip().str.upper()

    # Remove obvious duplicates
    df = df.drop_duplicates()

    # Reset index after filtering
    df = df.reset_index(drop=True)

    return df


In [65]:
# CAPSTONE: S3 → PANDAS → SNOWFLAKE PIPELINE

In [66]:
"""
Full end-to-end pipeline:
  - Reads raw CSV files from an S3 landing zone
  - Transforms data with pandas
  - Writes cleaned Parquet back to S3 (processed zone)
  - Loads final data into a Snowflake table
  - Uses structured logging and error handling
"""

'\nFull end-to-end pipeline:\n  - Reads raw CSV files from an S3 landing zone\n  - Transforms data with pandas\n  - Writes cleaned Parquet back to S3 (processed zone)\n  - Loads final data into a Snowflake table\n  - Uses structured logging and error handling\n'

In [67]:
import logging
from datetime import datetime, timezone


In [68]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)

In [16]:
from datetime import datetime, timezone
import io
import logging
import boto3
import pandas as pd

class S3ToSnowflakePipeline:
    """
    Orchestrates an ELT pipeline from S3 (raw CSV) to Snowflake.

    Usage:
        pipeline = S3ToSnowflakePipeline(
            raw_bucket="my-data-lake-raw",
            processed_bucket="my-data-lake-processed",
            raw_prefix="sales/incoming/",
            processed_prefix="sales/clean/",
            snowflake_table="SALES_FACT",
        )
        pipeline.run()
    """

    def __init__(
        self,
        raw_bucket,
        processed_bucket,
        raw_prefix,
        processed_prefix,
        snowflake_table,
        snowflake_database="ANALYTICS",
        snowflake_schema="PUBLIC",
    ):
        self.raw_bucket = raw_bucket
        self.processed_bucket = processed_bucket
        self.raw_prefix = raw_prefix
        self.processed_prefix = processed_prefix
        self.snowflake_table = snowflake_table
        self.snowflake_database = snowflake_database
        self.snowflake_schema = snowflake_schema
        self.run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        #self.s3 = boto3.client("s3")
        self.s3 = boto3.client(
            "s3",
            region_name="us-east-2",
            aws_access_key_id=AWS_ACCESS_KEY,
            aws_secret_access_key=AWS_SECRET_KEY,
        )

    # ── Step 1: Extract ──────────────────────────────────────────────

    def extract(self):
        """Read all CSV files from the raw S3 prefix into a single DataFrame."""
        log.info(f"[EXTRACT] Scanning s3://{self.raw_bucket}/{self.raw_prefix}")

        paginator = self.s3.get_paginator("list_objects_v2")
        pages = paginator.paginate(Bucket=self.raw_bucket, Prefix=self.raw_prefix)

        dfs = []
        file_count = 0
        for page in pages:
            for obj in page.get("Contents", []):
                key = obj["Key"]
                if not key.endswith(".csv"):
                    continue
                log.info(f"  Reading: {key}")
                response = self.s3.get_object(Bucket=self.raw_bucket, Key=key)
                df = pd.read_csv(io.BytesIO(response["Body"].read()))
                df["_source_file"] = key
                dfs.append(df)
                file_count += 1

        if not dfs:
            raise ValueError(f"No CSV files found at s3://{self.raw_bucket}/{self.raw_prefix}")

        raw_df = pd.concat(dfs, ignore_index=True)
        log.info(f"[EXTRACT] Loaded {file_count} file(s), {len(raw_df)} total rows.")
        return raw_df

    # ── Step 2: Transform ────────────────────────────────────────────

    def transform(self, df):
        """Clean and enrich raw data."""
        log.info(f"[TRANSFORM] Starting with {len(df)} rows.")

        initial_count = len(df)

        # Standardize column names
        df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)

        # Drop fully empty rows
        df = df.dropna(how="all")

        # Parse and validate dates
        df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
        invalid_dates = df["order_date"].isna().sum()
        if invalid_dates:
            log.warning(f"  Dropping {invalid_dates} rows with invalid dates.")
        df = df.dropna(subset=["order_date"])

        # Coerce numeric columns
        for col in ["quantity", "price", "total_amount"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df = df.dropna(subset=["quantity", "price", "total_amount"])

        # Derived metrics
        df["revenue"]    = (df["quantity"] * df["price"]).round(2)
        df["year"]       = df["order_date"].dt.year
        df["month"]      = df["order_date"].dt.month
        df["quarter"]    = df["order_date"].dt.quarter

        # Normalize text fields
        for col in ["product_name", "category"]:
            if col in df.columns:
                df[col] = df[col].str.strip().str.upper()

        # Dedup
        df = df.drop_duplicates()

        # Add pipeline metadata
        df["_pipeline_run_id"]  = self.run_id
        df["_loaded_at"]        = datetime.now(timezone.utc).isoformat()

        # Drop internal tracking column before loading
        df = df.drop(columns=["_source_file"], errors="ignore")

        df = df.reset_index(drop=True)
        log.info(f"[TRANSFORM] Complete. {initial_count} → {len(df)} rows after cleaning.")
        return df

    # ── Step 3a: Write processed data to S3 ─────────────────────────

    def write_to_s3(self, df):
        """Persist the cleaned DataFrame to the processed S3 zone as Parquet."""
        key = f"{self.processed_prefix}run_id={self.run_id}/data.parquet"
        log.info(f"[WRITE S3] Writing {len(df)} rows → s3://{self.processed_bucket}/{key}")

        buffer = io.BytesIO()
        df.to_parquet(buffer, index=False, engine="pyarrow")
        buffer.seek(0)
        self.s3.put_object(
            Bucket=self.processed_bucket,
            Key=key,
            Body=buffer.read(),
        )
        log.info(f"[WRITE S3] Done.")
        return key

    # ── Step 3b: Load into Snowflake ─────────────────────────────────

    def load_to_snowflake(self, df):
        """Append the cleaned DataFrame to the target Snowflake table."""
        log.info(f"[LOAD] Connecting to Snowflake...")
        conn = get_snowflake_connection()
        try:
            log.info(f"[LOAD] Writing {len(df)} rows → {self.snowflake_database}.{self.snowflake_schema}.{self.snowflake_table}")
            load_to_snowflake(
                conn=conn,
                df=df,
                table_name=self.snowflake_table,
                database=self.snowflake_database,
                schema=self.snowflake_schema,
                overwrite=False,
            )
        finally:
            conn.close()
            log.info("[LOAD] Snowflake connection closed.")

    # ── Orchestrator ─────────────────────────────────────────────────

    def run(self):
        """Execute the full pipeline: Extract → Transform → Write S3 → Load Snowflake."""
        log.info(f"{'='*55}")
        log.info(f"  Pipeline run started  |  run_id: {self.run_id}")
        log.info(f"{'='*55}")
        start = datetime.now(timezone.utc)

        try:
            raw_df       = self.extract()
            clean_df     = self.transform(raw_df)
            s3_output    = self.write_to_s3(clean_df)
            self.load_to_snowflake(clean_df)

            elapsed = (datetime.now(timezone.utc) - start).total_seconds()
            log.info(f"{'='*55}")
            log.info(f"  Pipeline SUCCESS  |  {len(clean_df)} rows loaded  |  {elapsed:.1f}s")
            log.info(f"  Processed file  : s3://{self.processed_bucket}/{s3_output}")
            log.info(f"  Snowflake table : {self.snowflake_database}.{self.snowflake_schema}.{self.snowflake_table}")
            log.info(f"{'='*55}")

        except Exception as e:
            log.error(f"Pipeline FAILED: {e}", exc_info=True)
            raise


In [75]:
if __name__ == "__main__":
    pipeline = S3ToSnowflakePipeline(
        raw_bucket        = "de-aws-snowflake-demo",
        processed_bucket  = "de-aws-snowflake-demo",
        raw_prefix        = "sales/",
        processed_prefix  = "sales_processed/",
        snowflake_table   = "SALES_FACT",
        snowflake_database= "MY_DB1",
        snowflake_schema  = "MY_SC1",
    )
    pipeline.run()

2026-03-05 09:46:17 [INFO] =======================================================
2026-03-05 09:46:17 [INFO]   Pipeline run started  |  run_id: 20260305T041617Z
2026-03-05 09:46:17 [INFO] =======================================================
2026-03-05 09:46:17 [INFO] [EXTRACT] Scanning s3://de-aws-snowflake-demo/sales/
2026-03-05 09:46:19 [INFO]   Reading: sales/sales_large.csv
2026-03-05 09:46:19 [INFO] [EXTRACT] Loaded 1 file(s), 1000 total rows.
2026-03-05 09:46:19 [INFO] [TRANSFORM] Starting with 1000 rows.
2026-03-05 09:46:19 [INFO] [TRANSFORM] Complete. 1000 → 1000 rows after cleaning.
2026-03-05 09:46:19 [INFO] [WRITE S3] Writing 1000 rows → s3://de-aws-snowflake-demo/sales_processed/run_id=20260305T041617Z/data.parquet
2026-03-05 09:46:20 [INFO] [WRITE S3] Done.
2026-03-05 09:46:20 [INFO] [LOAD] Connecting to Snowflake...
2026-03-05 09:46:20 [INFO] Snowflake Connector for Python Version: 4.2.0, Python Version: 3.11.9, Platform: Windows-10-10.0.26200-SP0
2026-03-05 09:46:20 

Loaded 1000 rows into MY_DB1.MY_SC1.SALES_FACT (1 chunk(s))
